# 20 · Gold de fonte única (G-U1 a G-U8)

Tabelas que um dataset já entrega sozinho: orçamento, calendário de polos, setores, faixas, CIS (gestor), encosta por bairro × mês, eventos de chuva, mapa base, perfil de serviço e a linha de base do 156.

Cada tabela leva `fonte_dado` e `uso` (*gestor* ou *publico*).

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "src" / "wastestack").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
from wastestack.arquivos import ler
from wastestack.config import SILVER, GOLD

In [2]:
from wastestack.gold import gold_fonte_unica
resumo = gold_fonte_unica()
resumo

{
 "limpeza_urbana_2024_mi": 309.3,
 "polos": 18,
 "cis_camada": 471,
 "cis_pequenas_agregadas": 74,
 "dias_com_alagamento": 17,
 "recorte": 61790,
 "recorte_meses_completos": 58342,
 "ruas_10_mais": 1927,
 "ruas_20_de_25_meses": 88,
 "drenagem_aberta": 9955,
 "drenagem_aberta_com_movimento_2026": 2612,
 "drenagem_parada_2024_2025": 7343,
 "recorte_aberto_com_movimento_2026": 5912
}


In [3]:
assert resumo["limpeza_urbana_2024_mi"] == 309.3 and resumo["polos"] == 18 and resumo["dias_com_alagamento"] == 17
rl = ler("base156_reincidencia_logradouro", camada=GOLD)
assert not rl.publicavel.any(), "sem NUMERO, a regra de 10+ endereços não é verificável"
print("Ruas com 10+ chamados:", resumo["ruas_10_mais"], "| ruas com chamado em 20+ dos 25 meses:", resumo["ruas_20_de_25_meses"], "(referência com NUMERO: 88)")
print("Drenagem aberta com movimento no arquivo de 2026:", resumo["drenagem_aberta_com_movimento_2026"], "(referência com NUMERO: 2.662)")
print("Drenagem não atendida e parada desde 2024–2025:", resumo["drenagem_parada_2024_2025"], "(referência com NUMERO: 7.494)")
print("Recorte aberto com movimento em 2026:", resumo["recorte_aberto_com_movimento_2026"], "(referência com NUMERO: 6.018)")

Ruas com 10+ chamados: 1927 | ruas com chamado em 20+ dos 25 meses: 88 (referência com NUMERO: 88)
Drenagem aberta com movimento no arquivo de 2026: 2612 (referência com NUMERO: 2.662)
Drenagem não atendida e parada desde 2024–2025: 7343 (referência com NUMERO: 7.494)
Recorte aberto com movimento em 2026: 5912 (referência com NUMERO: 6.018)


## Limites e cuidados

- **156 sem NUMERO:** não dá para contar endereços distintos. As tabelas de rua ficam com `publicavel = False` e uso interno do gestor; a regra não é contornada.
- CIS pequenas saem da camada nominal e ficam só no agregado por bairro.
- Defesa Civil: só bairro × mês e dia; nunca rua × dia.